In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

# 1. LOAD DATA
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 2. TARGET: Predict the difference (Delta)
# This centers the trees around a mean of 0, making them much more accurate
train['Target_Diff'] = train['Close'].shift(-1) - train['Close']

# 3. ENGINEERING HINT: 30-Day Indicators
def add_30d_indicators(df, is_train=True):
    if is_train:
        df['SMA_30'] = df['Close'].rolling(30).mean()
        df['Avg_Vol_30'] = df['Volume'].rolling(30).mean()
    else:
        c_cols = [f'Close_Lag_{i}' for i in range(30)]
        v_cols = [f'Volume_Lag_{i}' for i in range(30)]
        df['SMA_30'] = df[c_cols].mean(axis=1)
        df['Avg_Vol_30'] = df[v_cols].mean(axis=1)
    return df

train = add_30d_indicators(train)
test = add_30d_indicators(test, is_train=False)

# 4. GENERATE RECENT LAGS (Focusing trees on most relevant data)
# Using only 5 days for the raw lags reduces tree depth and overfitting
new_cols = {}
for feat in ['Open', 'High', 'Low', 'Close', 'Volume']:
    for lag in range(5):
        new_cols[f'{feat}_Lag_{lag}'] = train[feat].shift(lag)
train_final = pd.concat([train[['Target_Diff', 'SMA_30', 'Avg_Vol_30']], pd.DataFrame(new_cols)], axis=1).dropna()

# 5. FEATURE SELECTION
indicators = ['SMA_30', 'Avg_Vol_30']
feature_cols = [c for c in train_final.columns if '_Lag_' in c] + indicators

X_train = train_final[feature_cols].values
y_train = train_final['Target_Diff'].values
X_test = test[feature_cols].values

# 6. OPTIMIZED BAGGING (Random Forest)
# min_samples_leaf prevents the trees from memorizing specific noisy days
bag_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1 # Uses all CPU cores
)
bag_model.fit(X_train, y_train)

# 7. PREDICT AND RECONSTRUCT PRICE
predicted_diff = bag_model.predict(X_test)
final_prices = test['Close_Lag_0'].values + predicted_diff # Today + predicted change

# 8. THE 4% SAFETY CLIP
# This prevents wild forest predictions from blowing up your score
final_prices = np.clip(final_prices,
                       test['Close_Lag_0'].values * 0.96,
                       test['Close_Lag_0'].values * 1.04)

# 9. SUBMISSION
submission = pd.DataFrame({'ID': test['ID'], 'Target': final_prices})
submission.to_csv('submission.csv', index=False)
print("Optimized Bagging Submission Saved.")

Optimized Bagging Submission Saved.
